In [1]:
import torch
import torchvision.datasets
import torchvision.transforms

import torch.nn as nn
import torch.nn.functional as F
import torch.autograd.functional as AGF
import torch.optim as optim
import matplotlib.pyplot as plt

from math import prod
import numpy
import pandas as pd
import os

from AccNets4 import Net as Net4
from AccNets2 import Net as Net2
from trainAccNets3 import trainNet
from tqdm import tqdm


In [2]:
import numpy as np

# models = [hparams.model]
d_in = 15
d_full = 500
d_mid = 500
d_out = 20

L = 5

data_path = './sampled_kernels/'

widths = [d_in] + [d_full, d_mid] + [d_out]
Ns  = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000]

nan_value = pd.DataFrame(index = np.arange(10), columns = np.arange(10)).iloc[0,0]
num_dif = np.arange(20)/2 + 0.5

AccNets=False
model = 'dnn'
trials = 1

for indxN, N in enumerate(Ns):
    #L1L1_prof_w500_500_longw8
    cost_path = './test_cost/df_test_costs8_{N}_{model}_{name}.pkl'.format(model=model, N = N, name = 'final_test_noClass')
    try:
        df_test_costs = pd.read_pickle(cost_path)
    except:
        df_test_costs = pd.DataFrame(index = num_dif, columns = num_dif)
        df_test_costs.to_pickle(cost_path)
    
    vh = np.tile(num_dif,20).reshape(20,-1)[df_test_costs.isna()]
    vg = np.tile(num_dif,20).reshape(20,-1).T[df_test_costs.isna()]
    
    if vh.shape != vg.shape:
        print('the # of null in vh & vg unmatched. check df_test_costs file')
        break

    num_nan = vh.shape[0]
    
    for i in tqdm(range(num_nan)):
        
        model_path = './model_weights/Weight_{model}_N{N}_vg{vg}_vh{vh}_{name}.pth'.format(model = model, N = N, vg = vg[i], vh = vh[i], name = 'final_test')
        complexity_path = './complexity/comp_{model}_N{N}_{name}.pkl'.format(model=model, N = N, name = 'final_test')
        comp_param_path = './complexity/comp_param_{model}_N{N}_{name}.pkl'.format(model=model, N = N, name = 'final_test')
        
        df_test_costs = pd.read_pickle(cost_path)
        
        # if df_test_costs.isna().loc[vg[i],vh[i]]:
        if df_test_costs.isna().loc[vg[i],vh[i]]:
            X_path = 'X_g_Matern7_{nu_g}_{nu_h}_50000.pkl'.format(nu_g=vg[i], nu_h=vh[i])
            Y_path = 'Y_g_Matern7_{nu_g}_{nu_h}_50000.pkl'.format(nu_g=vg[i], nu_h=vh[i])
            X = torch.Tensor(pd.read_pickle(data_path + X_path)).cuda()
            Y = torch.Tensor(pd.read_pickle(data_path + Y_path)).cuda()
            
            df_test_costs.loc[vg[i],vh[i]] = 931230
            df_test_costs.to_pickle(cost_path)
            
            path = "_g_Matern7_{nu_g}_{nu_h}_50000.pkl".format(nu_g = vg[i], nu_h = vh[i])
            name = 'Matern7_{nu_g}_{nu_h}_{model}_{name}'.format(nu_g = vg[i], nu_h = vh[i], model = model, name = 'final_test')
            # NN = trainNet(path2data = path, prjt_name = name, Ns = [N], test_N = 2500, w = widths, loss = "L2",
            #               test_loss="L2", trial=3, prof="L2", wandb=False, 
            #               lr_scale = 1, AccNets = AccNets, w8_dk = 0, L=5)
            # NN.train_all()
        
            net4 = Net4(widths = widths, loss='L2', test_loss='L2', prof=False, AccNets = False, jac = True, print_progress = False, L = 5).cuda()
            net4.train(X[:N], Y[:N], X[N:N+2500], Y[N:N+2500], lr=1 * 0.001, weight_decay = 0, epochs=500, num_batches=5)
            net4.train(X[:N], Y[:N], X[N:N+2500], Y[N:N+2500], lr=0.3 * 0.001, weight_decay = 0.0001, epochs=500, num_batches=5)
            test_cost = net4.train(X[:N], Y[:N], X[N:N+2500], Y[N:N+2500], lr=0.1 * 0.001, weight_decay = 0.0002, epochs=2000, num_batches=5)
            
            torch.save(net4.state_dict(), model_path)
            df_test_costs = pd.read_pickle(cost_path)
            # df_test_costs.loc[vg[i],vh[i]] = [NN.test_costs, NN.train_costs]
            df_test_costs.loc[vg[i],vh[i]] = test_cost
            df_test_costs.to_pickle(cost_path)
            
              

0it [00:00, ?it/s]
100%|██████████| 397/397 [1:04:35<00:00,  9.76s/it]


In [2]:


vh = 10.0
vg = 10.0
X = torch.Tensor(pd.read_pickle('./sampled_kernels/X_g_Matern7_{nu_g}_{nu_h}_50000.pkl'.format(nu_g=vg, nu_h=vh))).cuda()
Y = torch.Tensor(pd.read_pickle('./sampled_kernels/Y_g_Matern7_{nu_g}_{nu_h}_50000.pkl'.format(nu_g=vg, nu_h=vh))).cuda()

In [8]:
N = 50000
test_N = 2500
model = 'acc'

net4 = Net4(widths = widths, loss='L2', test_loss='L2', prof=False, AccNets = True, jac = True, print_progress = True, L = 5).cuda()

# model_path = './model_weights/Weight_{model}_N{N}_vg{vg}_vh{vh}_testing_acc_25feb6.pth'.format(model = model, N = N, vg = vg, vh = vh)
# complexity_path = './complexity/comp_{model}_N{N}_testing_acc_25feb6.pkl'.format(model=model, N = N)
# comp_param_path = './complexity/comp_param_{model}_N{N}_testing_acc_25feb6.pkl'.format(model=model, N = N)

# path = "_g_Matern7_{nu_g}_{nu_h}_50000.pkl".format(nu_g = vg, nu_h = vh)
# name = 'Matern7_{nu_g}_{nu_h}_{model}_testing_acc_25feb6'.format(nu_g = vg, nu_h = vh, model = model)             

In [7]:
print('first')
net4.train(X[:N], Y[:N], X[N:N+2500], Y[N:N+2500],
          lr=1 * 0.001, weight_decay = 0, epochs=500, num_batches=5)
print('second')
net4.train(X[:N], Y[:N], X[N:N+2500], Y[N:N+2500],
          lr=0.3 * 0.001, weight_decay = 0.0001, epochs=500, num_batches=5)
print('third')
test_cost = net4.train(X[:N], Y[:N], X[N:N+2500], Y[N:N+2500],
          lr=0.1 * 0.001, weight_decay = 0.0002, epochs=2000, num_batches=5)

first
{'cost': '0.0177', 'test_cost': '0.0440', 'norm': '58.9872'}
{'cost': '0.0279', 'test_cost': '3.6652', 'norm': '59.1932'}
{'cost': '3.7353', 'test_cost': '2.1111', 'norm': '58.1540'}
{'cost': '2.1458', 'test_cost': '0.7380', 'norm': '57.8311'}
{'cost': '0.7541', 'test_cost': '0.7090', 'norm': '59.1310'}
{'cost': '0.1564', 'test_cost': '0.1568', 'norm': '216.5797'}
{'cost': '0.1556', 'test_cost': '0.1566', 'norm': '216.7421'}
{'cost': '0.1544', 'test_cost': '0.1554', 'norm': '216.8862'}
{'cost': '0.1539', 'test_cost': '0.1560', 'norm': '217.0295'}
{'cost': '0.1529', 'test_cost': '0.1548', 'norm': '217.2021'}
{'cost': '0.0795', 'test_cost': '0.0882', 'norm': '247.2976'}
{'cost': '0.0824', 'test_cost': '0.0994', 'norm': '247.4510'}
{'cost': '0.0918', 'test_cost': '0.1141', 'norm': '247.5756'}
{'cost': '0.1084', 'test_cost': '0.1159', 'norm': '247.6713'}
{'cost': '0.1068', 'test_cost': '0.1038', 'norm': '247.7108'}
{'cost': '0.0427', 'test_cost': '0.0533', 'norm': '273.0049'}
{'cost'

In [15]:
import numpy as np

# models = [hparams.model]
d_in = 15
d_full = 900
d_mid = 100
d_out = 20

L = 5

widths = [d_in] + [d_full, d_mid] + [d_out]
Ns  = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000]

nan_value = pd.DataFrame(index = np.arange(10), columns = np.arange(10)).iloc[0,0]
num_dif = np.arange(20)/2 + 0.5

AccNets=True

model = 'acc'

for indxN, N in enumerate(Ns):
    #L1L1_prof_w500_500_longw8
    cost_path = './test_cost/df_test_costs8_{N}_{model}_{name}.pkl'.format(model=model, N = N, name = 'final_test')
    try:
        df_test_costs = pd.read_pickle(cost_path)
    except:
        df_test_costs = pd.DataFrame(index = num_dif, columns = num_dif)
        df_test_costs.to_pickle(cost_path)
    
    vh = np.tile(num_dif,20).reshape(20,-1)[df_test_costs.isna()]
    vg = np.tile(num_dif,20).reshape(20,-1).T[df_test_costs.isna()]
    
    if vh.shape != vg.shape:
        print('the # of null in vh & vg unmatched. check df_test_costs file')
        break

    num_nan = vh.shape[0]
    
    for i in tqdm(range(num_nan)):
        
        model_path = './model_weights/Weight_{model}_N{N}_vg{vg}_vh{vh}_{name}.pth'.format(model = model, N = N, vg = vg[i], vh = vh[i], name = 'final_test')
        complexity_path = './complexity/comp_{model}_N{N}_{name}.pkl'.format(model=model, N = N, name = 'final_test')
        comp_param_path = './complexity/comp_param_{model}_N{N}_{name}.pkl'.format(model=model, N = N, name = 'final_test')
        
        df_test_costs = pd.read_pickle(cost_path)
        
        nu_g = 0.5 + vg[i]
        nu_h = 0.5 + vh[i]
        
        # if df_test_costs.isna().loc[vg[i],vh[i]]:
        if df_test_costs.isna().loc[vg[i],vh[i]]:
            
            df_test_costs.loc[vg[i],vh[i]] = 931230
            df_test_costs.to_pickle(cost_path)
            
            path = "_g_Matern7_{nu_g}_{nu_h}_50000.pkl".format(nu_g = vg[i], nu_h = vh[i])
            name = 'Matern7_{nu_g}_{nu_h}_{model}_{name}'.format(nu_g = vg[i], nu_h = vh[i], model = model, name = 'final_test')
            
            NN = trainNet(path2data = path, prjt_name = name, Ns = [N], test_N = 2500, w = widths, loss = "L2",
                          test_loss="L2", trial=3, prof="L2", wandb=False, 
                          lr_scale = 1, AccNets = AccNets, w8_dk = 0, L=5)
            NN.train_all()
            torch.save(NN.net.state_dict(), model_path)
    
            df_test_costs = pd.read_pickle(cost_path)
            df_test_costs.loc[vg[i],vh[i]] = [NN.test_costs, NN.train_costs]
            df_test_costs.to_pickle(cost_path)

            try:
                df_complexity = pd.read_pickle(complexity_path)
            except: 
                df_complexity = pd.DataFrame(index = num_dif, columns = num_dif)
                df_complexity.to_pickle(complexity_path)
                
            df_complexity.loc[vg[i],vh[i]] = NN.complexity#.detach().cpu().item()
            df_complexity.to_pickle(complexity_path)
            
            try:
                df_comp_param = pd.read_pickle(comp_param_path)
            except: 
                df_comp_param = pd.DataFrame(index = num_dif, columns = num_dif)
                df_comp_param.to_pickle(comp_param_path)
            
            df_comp_param.loc[vg[i],vh[i]] = [{"norms": NN.net.norms(), "Lip_ops": NN.net.Lip_OP(), 
                                               "ranks": NN.net.ranks(), 
                                               "complexity_min": NN.net.complexities(NN.X[:N])}]
            try:
                df_comp_param.to_pickle(comp_param_path)
            except:
                continue
            #     except:
            #         continue
            # except: 
            #     continue
                

  0%|          | 0/399 [00:40<?, ?it/s]


NameError: name 'torch' is not defined

In [14]:
torch

<module 'torch' from '/ext3/miniconda3/lib/python3.11/site-packages/torch/__init__.py'>